In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from transformers import (
    RobertaTokenizer, 
    RobertaForMaskedLM, 
    DataCollatorForLanguageModeling, 
    TrainingArguments,
    Trainer,
)

from datasets import load_dataset, concatenate_datasets
from embed_stage2 import prepare_token_dataset

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might invo

In [3]:
BLOCK_SIZE = 128 # Stride length when splitting long texts into 512-length segments
MAX_SEQ_LEN = 512 # maximum length of a sequence that BERT can operate on

posting_txt_col = 'description'
resume_txt_col = 'Resume_str'

In [4]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForMaskedLM.from_pretrained('roberta-base')

In [12]:
# posting_dataset = load_dataset('csv', data_files='./temp/postings5k.csv')
# resume_dataset = load_dataset('csv', data_files='./resume_data/Resume.csv')

posting_csv_path = './temp/postings5k.csv'
resume_csv_path = './resume_data/Resume.csv'

In [13]:
lm_dataset = prepare_token_dataset(tokenizer, posting_path=posting_csv_path, resume_path=resume_csv_path)

Map (num_proc=12):   0%|          | 0/6361 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1123 [00:00<?, ? examples/s]

In [14]:
lm_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 34006
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 5871
    })
})

In [15]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True
)

In [19]:
training_args = TrainingArguments(
    output_dir="./temp/roberta-tuned",
    overwrite_output_dir=True,
    logging_strategy='epoch',
    eval_strategy='epoch',
    num_train_epochs=25,
#     learning_rate=2e-5,
    per_device_train_batch_size=48,
#     save_steps=500,
    save_strategy='epoch',
    save_total_limit=1,
    seed=1
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_dataset['train'],
    eval_dataset=lm_dataset['test']
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.595800,1.362536
2,1.368300,1.311025
3,1.310900,1.300333
4,1.264400,1.272886
5,1.229700,1.250280
6,1.197200,1.236800
7,1.173400,1.222993
8,1.148400,1.219456
9,1.124400,1.210435
10,1.108200,1.204863


TrainOutput(global_step=17725, training_loss=1.1065998855397126, metrics={'train_runtime': 4401.6686, 'train_samples_per_second': 193.143, 'train_steps_per_second': 4.027, 'total_flos': 2.238151336210944e+17, 'train_loss': 1.1065998855397126, 'epoch': 25.0})

In [22]:
trainer.save_model('./roberta-tuned')

In [6]:
# def preprocess_posting(x):
#     '''preprocess for a job listing, as read from the linkedin CSV file'''
#     return tokenizer(x[posting_txt_col], add_special_tokens=True)

# def preprocess_resume(x):
#     '''preprocess for a resume CSV file'''
#     return tokenizer(x[resume_txt_col], add_special_tokens=True)


# tokenized_posting_dataset = posting_dataset.map(
#     preprocess_posting,
#     batched=True,
#     num_proc=12,
#     remove_columns=posting_dataset['train'].column_names,
# )

# tokenized_resume_dataset = resume_dataset.map(
#     preprocess_resume,
#     batched=True,
#     num_proc=12,
#     remove_columns=resume_dataset['train'].column_names,
# )

In [7]:
# tokenized_dataset = concatenate_datasets([tokenized_posting_dataset['train'], tokenized_resume_dataset['train']])
# tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.15)
# tokenized_dataset

In [40]:
# def group_texts(token_data):
#     '''
#     Split texts that are two long (longer than 512 tokens) into overlapping segments with a sliding window of stride 128,
#     and window size of 512. For example, a text with 1376 tokens is broken down into 8 segments that start at the 
#     following indices:
#         [0, 128, 256, 384, 512, 640, 768, 896]
#     Then each segment has size:
#         [512, 512, 512, 512, 512, 512, 512, 480]
#     '''
#     result = {k: [] for k in token_data.keys()}
    
#     for key, data in token_data.items():
#         for token_list in data:
#             n_tokens = len(token_list)
#             shifts = [t for t in range(0, n_tokens, BLOCK_SIZE) if t+MAX_SEQ_LEN-n_tokens < BLOCK_SIZE]
#             if len(shifts) == 0:
#                 shifts = [0]

#             for shift in shifts:
#                 result[key].append(token_list[shift : shift+MAX_SEQ_LEN])

#     return result
    
    

In [94]:
# lm_dataset = tokenized_dataset.map(group_texts, batched=True, num_proc=12)


In [25]:
# lm_dataset['train'].set_format('torch', columns=['input_ids', 'attention_mask'])

In [104]:
# from torch.utils.data import DataLoader
# loader = DataLoader(lm_dataset['train'],batch_size=32,collate_fn=tokenizer.pad,shuffle=False)

In [105]:
# t = next(iter(loader))
# np.array(t.attention_mask).sum(axis=1)

array([512, 512, 512, 512, 512, 512, 403, 512, 413, 346, 512, 512, 464,
       512, 512, 512, 512, 387, 420, 512, 497, 512, 512, 512, 512, 512,
       512, 512, 512, 495, 512, 512])

In [20]:
tokenizer.convert_ids_to_tokens(tokenizer.encode('Hi, my name is Keenan.\nHow are you?'))

['<s>',
 'Hi',
 ',',
 'Ġmy',
 'Ġname',
 'Ġis',
 'ĠKeen',
 'an',
 '.',
 'Ċ',
 'How',
 'Ġare',
 'Ġyou',
 '?',
 '</s>']

In [17]:
from transformers import pipeline
fill_mask = pipeline(
    "fill-mask",
    model="./roberta-tuned",
    tokenizer="roberta-base"
)
fill_mask("I am a highly motivated <mask>.")

Device set to use cuda:0


[{'score': 0.22024141252040863,
  'token': 2038,
  'token_str': ' professional',
  'sequence': 'I am a highly motivated professional.'},
 {'score': 0.1929740011692047,
  'token': 1736,
  'token_str': ' individual',
  'sequence': 'I am a highly motivated individual.'},
 {'score': 0.09850453585386276,
  'token': 621,
  'token_str': ' person',
  'sequence': 'I am a highly motivated person.'},
 {'score': 0.0949028730392456,
  'token': 869,
  'token_str': ' player',
  'sequence': 'I am a highly motivated player.'},
 {'score': 0.04590441659092903,
  'token': 3200,
  'token_str': ' employee',
  'sequence': 'I am a highly motivated employee.'}]

In [ ]:
dataset[0]

In [ ]:
len(dataset[0]['input_ids'])

In [ ]:
# import os
# import subprocess

# home = '/home/hice1/khom9'
# for d in os.listdir(home):
#     full_d = os.path.join(home, d)
#     if os.path.isdir(full_d):
# #         print(os.system(f'du -sh {full_d}'), full_d)
#         print(full_d)
#         res = subprocess.run(['du', '-sh', full_d], capture_output=True)
        

#         print(f'\t{str(res.stdout.decode("utf-8"))}')
    

In [1]:
import pandas as pd

df = pd.read_csv('./resume_data/Resume.csv')

In [2]:
df

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR
...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
